# router_final — training, gate post-hoc, deployment A3, evaluasi (Kaggle T4)

Satu seed per sesi Kaggle (batas 12 jam; VisDrone ±10–13 jam -> pakai `SESSION = 1`, `2`, lalu `3`, lihat sel 0). Urutan per seed: training → cek checkpoint → gate → ekspor → evaluasi →
suite final (H1–H7, latensi ujung-ke-ujung) → sensitivitas baseline `best.pt` → ablasi (seed 42 dan VisDrone saja).
Kode = folder `router_final/` di branch `final-arsitektur`, di atas **Ultralytics stok 8.3.240** (bukan fork di root repo).

**Melanjutkan di sesi baru:** unggah `arsip_<model>_seed<seed>.zip` sebagai dataset Kaggle, isi `RESUME_FROM`, lalu matikan
langkah yang sudah selesai di `STEPS`. Arsip disalin ke `/kaggle/working` supaya semua langkah bisa menulis ke `RUN`.
Engine TensorRT tidak ikut diarsip; langkah `eval` (sel 6) butuh `export` di sesi yang sama.

**Tiga sesi untuk VisDrone:** `SESSION = 1` = uji reproduksi + training + cek checkpoint + gate + ekspor + evaluasi TRT +
arsip (±10 jam; evaluasi PyTorch FP32 dilewati karena suite memeriksa FP32 pada seluruh test). Unggah arsip sebagai dataset,
lalu `SESSION = 2` + `RESUME_FROM` = cek NMS cepat + suite, dan `SESSION = 3` + `RESUME_FROM` = sensitivitas + ablasi.
Waktu sesi 2 dan 3 di VisDrone belum terukur (NMS & COCOeval pada scene padat jauh lebih lambat daripada KITTI).
`SESSION = None` = kendali manual lewat `STEPS`.

**NMS cepat (`fast_nms_k`, hanya VisDrone):** NMS dijalankan pada K kandidat teratas, dan hasilnya dipakai hanya bila
terbukti identik dengan NMS penuh (selain itu NMS penuh). Sel 6b memverifikasinya di val sebelum suite (gagal -> berhenti).

**Log training** tidak lagi memenuhi output notebook (progress bar membuat tahap simpan Kaggle sangat lama): stdout/stderr
masuk ke `/kaggle/working/logs/<nama>.log.gz`; notebook hanya mencetak epoch/mAP dari `results.csv` tiap 10 menit dan 25 baris
terakhir log. Perintah yang gagal sekarang menghentikan notebook (`!python` bawaan IPython mengabaikan kode keluar).


In [ ]:
# ===== 0. Kode dipin =====
REPO_URL = "https://github.com/maugi1547/thesis-Difficulty-Aware-Router"
CODE_COMMIT = "ISI_HASH_COMMIT"          # hash commit branch final-arsitektur yang dipakai (wajib, bukan nama branch)
SEED = 42                                # satu seed per sesi: 42, 7, 2026
CONFIG = "configs/kitti.yaml"
RUN_REPRO_CHECK = True                   # 2 × 2 epoch, bandingkan bobot (±30 menit). Wajib sekali per lingkungan baru.
# Mode sesi (None = kendali manual lewat STEPS di bawah). VisDrone > batas 12 jam -> tiga sesi:
#   1 = uji reproduksi + training + cek checkpoint + gate + ekspor + evaluasi TRT + arsip   (±10 jam terukur)
#   2 = cek NMS cepat + suite, dari arsip sesi 1 (isi RESUME_FROM)                         (belum terukur)
#   3 = sens_best + ablasi, dari arsip sesi 1 (isi RESUME_FROM)                             (belum terukur)
#   "baselines" = sel 3b saja (latih Vanilla + P2-Static, lalu berhenti)
SESSION = None
# Langkah per sesi. Sesi lanjutan: RESUME_FROM = folder arsip di /kaggle/input, dan set langkah yang sudah selesai = False.
STEPS = dict(baselines=False,               # sel 3b: latih Vanilla + P2-Static di sesi ini (VisDrone T4), lalu berhenti
             train=True, gate=True, export=True, eval=True,
             eval_fp32=True,                 # evaluasi PyTorch FP32 terpisah (VisDrone ±2 jam; suite sudah memeriksa FP32)
             nms_check=False,                # sel 6b: cek fast_nms_k (hanya config yang memakainya)
             suite=True,
             sens_best=True,                 # sensitivitas: suite ulang dengan baseline best.pt (±30 menit)
             ablation=SEED == 42)            # ablasi + uji berpasangan (±40 menit): seed 42 KITTI + VisDrone
_OFF = dict(baselines=False, train=False, gate=False, export=False, eval=False, eval_fp32=False, nms_check=False,
            suite=False, sens_best=False, ablation=False)
_PRESET = {"baselines": dict(_OFF, baselines=True),
           1: dict(_OFF, train=True, gate=True, export=True, eval=True),
           2: dict(_OFF, nms_check=True, suite=True),
           3: dict(_OFF, sens_best=True, ablation=SEED == 42)}
if SESSION is not None:
    STEPS = _PRESET[SESSION]                 # menimpa STEPS manual di atas
RESUME_FROM = None                       # mis. "/kaggle/input/arsip-kitti-seed7" (berisi weights/last.pt); None = training di sesi ini
# Baseline stok. last.pt = aturan sama dengan router (Tahap 4/4B lama memakai best.pt; angka baseline tidak sebanding
# dengan tabel lama). Sensitivitas best.pt otomatis di sel 7b (STEPS["sens_best"]).
# KITTI: checkpoint lama (Kaggle). VisDrone: DILATIH ULANG di T4 (sel 3b, sesi tersendiri ±8–9 jam untuk keduanya),
# lalu arsipnya diunggah sebagai dataset Kaggle dan path di bawah diisi.
BASELINES = {
    "configs/kitti.yaml": dict(
        vanilla="/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/last.pt",
        p2static="/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/last.pt"),
    "configs/visdrone.yaml": dict(
        vanilla="ISI_PATH/visdrone_vanilla_seed42/weights/last.pt",        # hasil sel 3b (T4)
        p2static="ISI_PATH/visdrone_p2static_seed42/weights/last.pt"),
}
assert CODE_COMMIT != "ISI_HASH_COMMIT", "Isi CODE_COMMIT"
assert not (STEPS["train"] and RESUME_FROM), "RESUME_FROM diisi -> STEPS['train'] harus False"
assert STEPS["train"] or STEPS["baselines"] or RESUME_FROM, "tanpa training di sesi ini: isi RESUME_FROM (arsip sesi 1)"
if STEPS["suite"] or STEPS["sens_best"]:
    assert "ISI_PATH" not in " ".join(BASELINES[CONFIG].values()), "isi BASELINES untuk CONFIG ini (path bobot baseline)"

import os, subprocess
REPO = "/kaggle/working/thesis-Difficulty-Aware-Router"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--all"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", CODE_COMMIT], check=True)
RF = f"{REPO}/router_final"
os.chdir(RF)                              # PENTING: jangan di root repo (folder ultralytics/ fork akan menutupi paket pip)
!pip install -q --no-deps ultralytics==8.3.240
!pip install -q onnx onnxscript onnxruntime pycocotools tensorrt
print(subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True))


# ---- Helper: jalankan perintah dengan log ke file (bukan ke output notebook) ----
import csv, glob, gzip, shutil, signal, time


def _tail(path, n=25, nbytes=65536):
    with open(path, "rb") as f:
        f.seek(0, 2); f.seek(max(0, f.tell() - nbytes))
        txt = f.read().decode("utf-8", "replace")
    return "\n".join(txt.replace("\r", "\n").splitlines()[-n:])      # progress bar memakai \r


def _last_result(results_glob):
    """Epoch, waktu, dan mAP50 dari baris terakhir results.csv (None bila belum ada / sedang ditulis)."""
    for p in sorted(glob.glob(results_glob)):
        try:
            rows = list(csv.DictReader(open(p)))
            if rows:
                r = {k.strip(): v.strip() for k, v in rows[-1].items()}
                return ", ".join(f"{k}={r[k]}" for k in r if k in ("epoch", "time") or k.startswith("metrics/mAP50"))
        except Exception:
            pass
    return None


def run_logged(cmd, name, results_glob=None, every_min=10, tail_lines=25, check=True):
    """
    Jalankan `cmd` (shell) dengan stdout+stderr -> /kaggle/working/logs/<name>.log (dikompres bila sukses).
    Notebook hanya mencetak ringkasan results.csv tiap `every_min` menit dan `tail_lines` baris terakhir log.
    check=True: kode keluar != 0 -> RuntimeError (notebook berhenti). Returns kode keluar.
    """
    os.makedirs("/kaggle/working/logs", exist_ok=True)
    log = f"/kaggle/working/logs/{name}.log"
    t0 = time.time()
    print(f"[{name}] mulai | log: {log}", flush=True)
    with open(log, "wb") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                             env=dict(os.environ, PYTHONUNBUFFERED="1"))
        try:
            while True:
                try:
                    rc = p.wait(timeout=every_min * 60)
                    break
                except subprocess.TimeoutExpired:
                    last = _last_result(results_glob) if results_glob else None
                    print(f"[{name}] {(time.time() - t0) / 3600:.2f} jam | {last or 'results.csv belum ada'}", flush=True)
        except BaseException:
            os.killpg(p.pid, signal.SIGKILL); p.wait(); raise
    print(f"[{name}] selesai, kode keluar {rc}, {(time.time() - t0) / 60:.1f} menit")
    print(_tail(log, tail_lines))
    if rc != 0:
        msg = f"{name}: kode keluar {rc} (log lengkap: {log})"
        if check:
            raise RuntimeError(msg)
        print("PERINGATAN:", msg)
        return rc
    with open(log, "rb") as src, gzip.open(log + ".gz", "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.remove(log)
    return rc


In [ ]:
# ===== 1. Uji unit (CPU, <1 menit): ultralytics stok, graf, loss, gate, B2, ONNX A3 =====
!python -m pytest -q -p no:cacheprovider tests

In [ ]:
# ===== 2. Uji reproduksi GPU: resep sama dua kali -> bobot harus identik bit per bit =====
if RUN_REPRO_CHECK and STEPS["train"]:          # hanya sesi yang melatih; BERBEDA -> peringatan, notebook lanjut
    run_logged(f"python scripts/check_reproducibility.py --config {CONFIG} --seed {SEED} --override epochs=2 "
               "project=/kaggle/working/repro", "repro", tail_lines=40, check=False)

In [ ]:
# ===== 3b. Baseline Vanilla + P2-Static: resep `train` yang SAMA, GPU yang sama (T4), seed yang sama =====
# Satu sesi khusus (±3 + ±5 jam VisDrone). Setelah selesai: unggah arsip sebagai dataset, isi BASELINES, matikan langkah ini.
if STEPS["baselines"]:
    import yaml, shutil, pathlib
    _c = yaml.safe_load(open(CONFIG))
    for which in ("vanilla", "p2static"):
        run = pathlib.Path(f"{_c['train']['project']}/{_c['name']}_{which}_seed{SEED}")
        run_logged(f"python scripts/train.py --config {CONFIG} --seed {SEED} --model {which}", run.name,
                   results_glob=f"{run}*/results.csv")
        print(run, open(run / "state_hash.json").read())
        shutil.make_archive(f"/kaggle/working/arsip_{run.name}", "zip", run.parent, run.name)
    raise SystemExit("Baseline selesai — unggah arsip_*.zip, isi BASELINES, set STEPS['baselines']=False untuk sesi router.")

In [ ]:
# ===== 3. Training detektor (100 epoch) atau lanjut dari arsip =====
import yaml, glob, shutil, pathlib
cfg = yaml.safe_load(open(CONFIG))
RUN = f"{cfg['train']['project']}/{cfg['name']}_seed{SEED}"
if STEPS["train"]:
    run_logged(f"python scripts/train.py --config {CONFIG} --seed {SEED}", f"{cfg['name']}_seed{SEED}",
               results_glob=f"{RUN}*/results.csv")
    # last.pt vs best.pt (best = fitness Branch A saja) di VAL, A dan B + bootstrap; laporan saja, last.pt tetap dipakai
    !python scripts/check_checkpoint.py --config {CONFIG} --run {RUN} --out {RUN}/cek_checkpoint
else:
    src = [pathlib.Path(p).parent.parent for p in glob.glob(f"{RESUME_FROM}/**/weights/last.pt", recursive=True)]
    assert len(src) == 1, f"arsip harus berisi tepat satu weights/last.pt: {src}"
    shutil.copytree(src[0], RUN, dirs_exist_ok=True)              # /kaggle/input read-only -> salin ke working
    print("lanjut dari", src[0])
print(RUN); print(open(f"{RUN}/state_hash.json").read())
assert f"seed{SEED}" in RUN and str(cfg["name"]) in RUN

In [ ]:
# ===== 4. Gate post-hoc (train) + thr0 (val) =====
if STEPS["gate"]:
    !python scripts/fit_gate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/gate

In [ ]:
# ===== 5. Ekspor A3 (ONNX If FP16) + verifikasi ORT + engine TensorRT =====
if STEPS["export"]:
    !python scripts/export.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/deploy --trt

In [ ]:
# ===== 6. Evaluasi test: TensorRT (sistem deploy) + PyTorch FP32 (pemeriksaan silang) =====
if STEPS["eval"]:
    !python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_trt --backend trt --engine {RUN}/deploy/a3_fp16.engine
    if STEPS["eval_fp32"]:
        !python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_pytorch --backend pytorch

In [ ]:
# ===== 6b. Cek NMS cepat (fast_nms_k) sebelum suite: deteksi harus IDENTIK dengan NMS penuh =====
# 300 gambar val × Branch A/B × FP32/FP16 × max_det gate/analisis × K {2000, 5000, 10000, config}. Juga mengukur
# percepatan & tingkat fallback. Tidak identik -> notebook berhenti sebelum suite (laporkan ringkasan.csv).
_k = (yaml.safe_load(open(CONFIG)).get("analysis") or {}).get("fast_nms_k")
if STEPS["nms_check"] and _k:
    run_logged(f"python scripts/check_fast_nms.py --config {CONFIG} --weights {RUN}/weights/last.pt "
               f"--out {RUN}/cek_nms --n 300 --assert-config", "cek_nms", tail_lines=45)
elif STEPS["nms_check"]:
    print("fast_nms_k tidak diset di", CONFIG, "-> NMS penuh, cek dilewati")


In [ ]:
# ===== 7. Evaluasi final terkunci + latensi ujung-ke-ujung (H1–H7, bootstrap, baseline, heuristik, kaskade) =====
# ±30–90 menit T4: cache train/val/test + 2 build engine + pass akurasi + latensi komponen + ujung-ke-ujung + bootstrap
B = BASELINES[CONFIG]
if STEPS["suite"]:
    !python scripts/suite.py run --config {CONFIG} --seed {SEED} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json \
        --vanilla "{B['vanilla']}" --p2static "{B['p2static']}" --out /kaggle/working/suite --zip /kaggle/working

In [ ]:
# ===== 7b. Sensitivitas: baseline best.pt (router, gate, dan engine router tetap) =====
# Tanpa --reuse: cache intermediate berisi deteksi baseline last.pt. Nama terpisah -> tidak tercampur saat merge.
BB = {k: v.replace("/weights/last.pt", "/weights/best.pt") for k, v in BASELINES[CONFIG].items()}
NAME_BB = f"{cfg['name']}_seed{SEED}_baseline_best"
if STEPS["sens_best"]:
    assert all(os.path.exists(p) for p in BB.values()), BB
    !python scripts/suite.py run --config {CONFIG} --seed {SEED} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json \
        --vanilla "{BB['vanilla']}" --p2static "{BB['p2static']}" --name {NAME_BB} --out /kaggle/working/suite --zip /kaggle/working

In [ ]:
# ===== 8. Ablasi + uji berpasangan (G1 target, G3 fitur, G4 classifier, G5 label, B anggaran) =====
# Menulis RUN/ablasi/{ablasi.csv, ablasi_uji.csv, cache_ablasi.pkl}; analisis ulang tanpa GPU: tambah --reuse-cache
if STEPS["ablation"]:
    !python scripts/ablate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/ablasi

In [ ]:
# ===== 9. Arsip (tanpa engine & ONNX besar) — unduh ketiganya: arsip + suite_* (utama & baseline_best) =====
keep = pathlib.Path(f"/kaggle/working/arsip_{cfg['name']}_seed{SEED}")
shutil.copytree(RUN, keep, ignore=shutil.ignore_patterns("*.engine", "*.onnx", "epoch*.pt"), dirs_exist_ok=True)
shutil.make_archive(str(keep), "zip", keep)
print(keep.with_suffix(".zip"))
print(sorted(str(p) for p in pathlib.Path("/kaggle/working").glob("suite_*.zip")))   # hasil + intermediate evaluasi final

## Ringkasan gabungan (sesi terakhir; CPU cukup)
Unggah semua `suite_<model>.zip` dan `suite_<model>_baseline_best.zip` (bukan `_intermediate`) sebagai satu dataset Kaggle,
lalu jalankan sel ini saja (setelah sel 0). Hasil utama dan sensitivitas digabung terpisah: model dipilih lewat namanya,
jadi folder `_baseline_best` tidak ikut pooling utama.

In [ ]:
# ===== 10. Gabung: utama (baseline last.pt) dan sensitivitas (baseline best.pt) — tabel kriteria, pooling KITTI 3 seed, H3 + Holm, Pareto =====
import shutil
MERGE_GLOB = "/kaggle/input/**/suite_*"     # sesuaikan dengan nama dataset hasil
MODELS = ["kitti_seed42", "kitti_seed7", "kitti_seed2026", "visdrone_seed42"]
MODELS_BB = " ".join(m + "_baseline_best" for m in MODELS)
MODELS = " ".join(MODELS)
!python scripts/suite.py merge --out /kaggle/working/suite_gabung --glob "{MERGE_GLOB}" --models {MODELS}
!python scripts/suite.py merge --out /kaggle/working/suite_gabung_baseline_best --glob "{MERGE_GLOB}" --models {MODELS_BB}
for d in ("suite_gabung", "suite_gabung_baseline_best"):
    print(shutil.make_archive(f"/kaggle/working/{d}", "zip", f"/kaggle/working/{d}"))